In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/subihanbiswas/socofing-metadata/metadata_204_persons.csv")

print(df.shape)
df.head()

(19723, 8)


,image_path,person_id,gender,finger,difficulty,alteration,finger_identity,finger_id
0,/kaggle/input/datasets/ruizgara/socofing/SOCOF...,463,NaN,F_Right_ring,original,none,463_F_Right_ring,1248
1,/kaggle/input/datasets/ruizgara/socofing/SOCOF...,334,NaN,F_Left_thumb,original,none,334_F_Left_thumb,764
2,/kaggle/input/datasets/ruizgara/socofing/SOCOF...,20,NaN,M_Right_little,original,none,20_M_Right_little,346
3,/kaggle/input/datasets/ruizgara/socofing/SOCOF...,86,NaN,M_Left_middle,original,none,86_M_Left_middle,1952
4,/kaggle/input/datasets/ruizgara/socofing/SOCOF...,434,NaN,M_Left_thumb,original,none,434_M_Left_thumb,1144


In [3]:
person_to_class = {
    person: idx
    for idx, person in enumerate(
        sorted(df.person_id.unique())
    )
}

df["person_class"] = (
    df.person_id.map(person_to_class)
)

In [4]:
print(df.person_class.nunique())

204


In [5]:
class_counts = df.groupby("person_class").size()

print(class_counts.min())
print(class_counts.max())

92
100


In [6]:
import numpy as np

np.random.seed(42)

new_rows = []

for cls in sorted(df.person_class.unique()):

    class_df = df[df.person_class == cls].copy()

    # If fewer than 100 images, randomly duplicate
    if len(class_df) < 100:

        extra_needed = 100 - len(class_df)

        extra_df = class_df.sample(
            n=extra_needed,
            replace=True,
            random_state=42
        )

        class_df = pd.concat(
            [class_df, extra_df],
            ignore_index=True
        )

    # Shuffle
    class_df = class_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    # Assign splits
    class_df.loc[:69, "split"] = "train"
    class_df.loc[70:84, "split"] = "val"
    class_df.loc[85:99, "split"] = "test"

    new_rows.append(class_df)

df_split = pd.concat(
    new_rows,
    ignore_index=True
)

In [7]:
train_df = df_split[df_split.split == "train"]
val_df = df_split[df_split.split == "val"]
test_df = df_split[df_split.split == "test"]

print(train_df.person_class.nunique())
print(val_df.person_class.nunique())
print(test_df.person_class.nunique())

204
204
204


In [9]:
print(df_split.split.value_counts())

split
train    14280
val       3060
test      3060
Name: count, dtype: int64


In [10]:
class_counts = df.groupby("person_class").size()

print(class_counts.min())
print(class_counts.max())

92
100


In [12]:
df_split.to_csv(
    "metadata_204_closedset_split_new.csv",
    index=False
)